In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/predicting-mobile-game-success/predicting_mobile_game_success_train_set.csv


In [2]:
import pandas as pd

train_url = "https://raw.githubusercontent.com/AhmedSalamaCs2020/game/main/predicting_mobile_game_success_train_set.csv"
test_url  = "https://raw.githubusercontent.com/AhmedSalamaCs2020/game/main/samples_mobile_game_success_test_set.csv"



In [3]:
# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np

# Model selection & evaluation
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# Preprocessing
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer

# Models
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
import lightgbm as lgb
import xgboost as xgb

# For combining sparse TFIDF with numeric features
from scipy.sparse import hstack


In [4]:
# ============================================================
# 2. LOAD DATA
# ============================================================
train_url = "https://raw.githubusercontent.com/AhmedSalamaCs2020/game/main/predicting_mobile_game_success_train_set.csv"
test_url  = "https://raw.githubusercontent.com/AhmedSalamaCs2020/game/main/samples_mobile_game_success_test_set.csv"

train = pd.read_csv(train_url)
test = pd.read_csv(test_url)

# Remove unwanted "Unnamed" columns (common CSV issue)
train = train.loc[:, ~train.columns.str.contains("^Unnamed")]
test = test.loc[:, ~test.columns.str.contains("^Unnamed")]

print("Train shape:", train.shape)
print("Test shape:", test.shape)


Train shape: (13599, 18)
Test shape: (6, 17)


In [5]:
# ============================================================
# 3. TARGET CLEANING
# ============================================================

# Convert target to numeric (in case it was read as string)
y = pd.to_numeric(train["Average User Rating"], errors="coerce")

# Drop target from train features
train = train.drop(columns=["Average User Rating"])

# Remove rows where target is missing
mask = y.notna()
train = train[mask]
y = y[mask]

# Reset index to avoid misalignment
train = train.reset_index(drop=True)
y = y.reset_index(drop=True)

print("Final training rows:", train.shape[0])
print("Missing target values:", y.isna().sum())


Final training rows: 6518
Missing target values: 0


In [6]:
# Convert price to numeric
train["Price"] = pd.to_numeric(train["Price"], errors="coerce")
test["Price"] = pd.to_numeric(test["Price"], errors="coerce")

# Binary feature: is the app free?
train["is_free"] = (train["Price"] == 0).astype(int)
test["is_free"] = (test["Price"] == 0).astype(int)


In [7]:
# Extract number of in-app purchases and average purchase price

def process_iap(column):
    counts = []
    averages = []
    
    for value in column.fillna(""):
        if value == "":
            counts.append(0)
            averages.append(0)
        else:
            prices = [float(x.strip()) for x in value.split(",")]
            counts.append(len(prices))
            averages.append(np.mean(prices))
    
    return counts, averages

train["iap_count"], train["iap_avg"] = process_iap(train["In-app Purchases"])
test["iap_count"], test["iap_avg"] = process_iap(test["In-app Purchases"])


In [8]:
train["Size_MB"] = train["Size"] / (1024 * 1024)
test["Size_MB"] = test["Size"] / (1024 * 1024)


In [9]:
train["lang_count"] = train["Languages"].fillna("").apply(lambda x: len(x.split(",")))
test["lang_count"] = test["Languages"].fillna("").apply(lambda x: len(x.split(",")))


In [10]:
train["Age Rating"] = train["Age Rating"].str.replace("+", "", regex=False)
test["Age Rating"] = test["Age Rating"].str.replace("+", "", regex=False)

train["Age Rating"] = pd.to_numeric(train["Age Rating"], errors="coerce")
test["Age Rating"] = pd.to_numeric(test["Age Rating"], errors="coerce")


In [11]:
# Convert to datetime
train["Original Release Date"] = pd.to_datetime(train["Original Release Date"], dayfirst=True, errors="coerce")
test["Original Release Date"] = pd.to_datetime(test["Original Release Date"], dayfirst=True, errors="coerce")

train["Current Version Release Date"] = pd.to_datetime(train["Current Version Release Date"], dayfirst=True, errors="coerce")
test["Current Version Release Date"] = pd.to_datetime(test["Current Version Release Date"], dayfirst=True, errors="coerce")

# App lifetime in days
train["app_age_days"] = (train["Current Version Release Date"] - train["Original Release Date"]).dt.days
test["app_age_days"] = (test["Current Version Release Date"] - test["Original Release Date"]).dt.days


In [12]:
train["User Rating Count"] = np.log1p(train["User Rating Count"])
test["User Rating Count"] = np.log1p(test["User Rating Count"])


In [13]:
# Encode Primary Genre and Developer using LabelEncoder

cat_cols = ["Primary Genre", "Developer"]

for col in cat_cols:
    le = LabelEncoder()
    combined = pd.concat([train[col], test[col]], axis=0).astype(str)
    le.fit(combined)
    train[col] = le.transform(train[col].astype(str))
    test[col] = le.transform(test[col].astype(str))


In [14]:
# ============================================================
# TEXT FEATURE ENGINEERING (CORRECT WAY)
# ============================================================

# Fill missing descriptions
train["Description"] = train["Description"].fillna("")
test["Description"] = test["Description"].fillna("")

# Create description length feature (numeric feature)
train["desc_length"] = train["Description"].apply(len)
test["desc_length"] = test["Description"].apply(len)

# Apply TF-IDF on actual description text
tfidf = TfidfVectorizer(max_features=300, stop_words="english")

train_text = tfidf.fit_transform(train["Description"])
test_text = tfidf.transform(test["Description"])


In [15]:
# Store test IDs for submission
test_ids = test["ID"]

drop_cols = [
    "ID", "Name", "Subtitle", "Icon URL", "URL",
    "Description", "Languages", "Genres",
    "Original Release Date", "Current Version Release Date",
    "In-app Purchases", "Size"
]

train = train.drop(columns=drop_cols)
test = test.drop(columns=drop_cols)


In [16]:
# Replace infinite values
train = train.replace([np.inf, -np.inf], np.nan)
test = test.replace([np.inf, -np.inf], np.nan)

# Fill numeric columns with median (robust to outliers)
num_cols = train.select_dtypes(include=[np.number]).columns

for col in num_cols:
    median = train[col].median()
    train[col] = train[col].fillna(median)
    test[col] = test[col].fillna(median)


In [17]:
X = hstack([train.values, train_text])
X_test = hstack([test.values, test_text])


In [18]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [19]:
import lightgbm as lgb

lgb_model = lgb.LGBMRegressor(
    n_estimators=2000,
    learning_rate=0.03,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

lgb_model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="rmse",
    callbacks=[
        lgb.early_stopping(stopping_rounds=100),
        lgb.log_evaluation(0)   # disables logging
    ]
)



[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.013178 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 46490
[LightGBM] [Info] Number of data points in the train set: 5214, number of used features: 303
[LightGBM] [Info] Start training from score 4.032892
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[175]	valid_0's rmse: 0.671818	valid_0's l2: 0.45134


LGBMRegressor(colsample_bytree=0.8, learning_rate=0.03, n_estimators=2000,
              n_jobs=-1, random_state=42, subsample=0.8)

In [20]:
print("Best iteration:", lgb_model.best_iteration_)


Best iteration: 175


In [21]:
preds = lgb_model.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, preds))
print("RMSE:", rmse)


RMSE: 0.6718181298966094


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [22]:
final_model = lgb.LGBMRegressor(
    n_estimators=lgb_model.best_iteration_,
    learning_rate=0.03,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

final_model.fit(X, y)

predictions = final_model.predict(X_test)
predictions = np.clip(predictions, 0, 5)


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.015396 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 53797
[LightGBM] [Info] Number of data points in the train set: 6518, number of used features: 303
[LightGBM] [Info] Start training from score 4.031221


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/lightgbm/basic.py:1238: UserWarning: Converting data to scipy sparse matrix.
  _log_warning("Converting data to scipy sparse matrix.")


In [23]:
submission = pd.DataFrame({
    "ID": test_ids,
    "Average User Rating": predictions
})

submission.to_csv("submission.csv", index=False)